# Step 3 - Embed and Index Knowledge Base

Loads chunked website data, generates embeddings, and stores vectors in a persistent ChromaDB collection.

In [1]:
from pathlib import Path
import json
import time

import chromadb
from sentence_transformers import SentenceTransformer

ROOT = Path.cwd()
if ROOT.name.lower() == "backend":
    ROOT = ROOT.parent

CHUNKS_FILE = ROOT / "knowledge_base/chunks.json"
CHROMA_DIR = ROOT / "backend/chroma_data"
COLLECTION_NAME = "website_knowledge"
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
BATCH_SIZE = 64

print(f"[INDEX] Chunks file: {CHUNKS_FILE}")
print(f"[INDEX] Chroma path: {CHROMA_DIR}")
print(f"[INDEX] Collection: {COLLECTION_NAME}")
print(f"[INDEX] Model: {EMBED_MODEL_NAME}")

c:\Users\danis\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[INDEX] Chunks file: d:\Users\Danish\Desktop\Projects\Attendance-Web\knowledge_base\chunks.json
[INDEX] Chroma path: d:\Users\Danish\Desktop\Projects\Attendance-Web\backend\chroma_data
[INDEX] Collection: website_knowledge
[INDEX] Model: sentence-transformers/all-MiniLM-L6-v2


In [6]:
if not CHUNKS_FILE.exists():
    raise FileNotFoundError(f"Missing chunks file: {CHUNKS_FILE}")

chunks = json.loads(CHUNKS_FILE.read_text(encoding="utf-8"))
if not isinstance(chunks, list) or not chunks:
    raise ValueError("chunks.json is empty or invalid")

required_keys = {"id", "content", "url", "source_file", "page_name", "section_num"}
for idx, item in enumerate(chunks):
    missing = required_keys - set(item.keys())
    if missing:
        raise ValueError(f"Chunk at index {idx} missing keys: {sorted(missing)}")

print(f"[INDEX] Loaded chunks: {len(chunks)}")
print(f"[INDEX] First id: {chunks[0]['id']}")
print(f"[INDEX] Last id: {chunks[-1]['id']}")

[INDEX] Loaded chunks: 25
[INDEX] First id: chunk_0000
[INDEX] Last id: chunk_0024


In [7]:
CHROMA_DIR.mkdir(parents=True, exist_ok=True)
client = chromadb.PersistentClient(path=str(CHROMA_DIR))

try:
    client.delete_collection(COLLECTION_NAME)
    print(f"[INDEX] Cleared existing collection: {COLLECTION_NAME}")
except Exception:
    print(f"[INDEX] No existing collection to clear: {COLLECTION_NAME}")

collection = client.get_or_create_collection(name=COLLECTION_NAME)
model = SentenceTransformer(EMBED_MODEL_NAME)

print("[INDEX] Chroma collection ready")
print("[INDEX] Embedding model loaded")

[INDEX] Cleared existing collection: website_knowledge


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2742.98it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[INDEX] Chroma collection ready
[INDEX] Embedding model loaded


In [8]:
def build_metadata(chunk: dict) -> dict:
    return {
        "url": str(chunk.get("url", "")),
        "source_file": str(chunk.get("source_file", "")),
        "page_name": str(chunk.get("page_name", "")),
        "section_num": int(chunk.get("section_num", 0)),
        "section_heading": str(chunk.get("section_heading", "")),
    }

start = time.time()
documents = [item["content"] for item in chunks]
ids = [item["id"] for item in chunks]
metadatas = [build_metadata(item) for item in chunks]

for i in range(0, len(documents), BATCH_SIZE):
    batch_docs = documents[i:i + BATCH_SIZE]
    batch_ids = ids[i:i + BATCH_SIZE]
    batch_meta = metadatas[i:i + BATCH_SIZE]

    embeddings = model.encode(batch_docs, normalize_embeddings=True).tolist()
    collection.add(
        ids=batch_ids,
        documents=batch_docs,
        metadatas=batch_meta,
        embeddings=embeddings,
    )

elapsed = time.time() - start
print(f"[INDEX] Indexed documents: {len(documents)}")
print(f"[INDEX] Time taken: {elapsed:.2f}s")

[INDEX] Indexed documents: 25
[INDEX] Time taken: 2.86s


In [9]:
total = collection.count()
print(f"[INDEX] Collection count: {total}")

probe = "attendance and leave management features"
probe_embedding = model.encode([probe], normalize_embeddings=True).tolist()
result = collection.query(
    query_embeddings=probe_embedding,
    n_results=min(3, total),
)

print("[INDEX] Retrieval smoke test complete")
for i, doc in enumerate(result.get("documents", [[]])[0], start=1):
    preview = doc[:140].replace("\n", " ")
    print(f"  {i}. {preview}...")

[INDEX] Collection count: 25
[INDEX] Retrieval smoke test complete
  1. Starter Plan  Best for small teams  Start with essential attendance operations and simple policy controls.  Attendance Tracking Leave Manage...
  2. MANO-Attendance Empowering Workforces  AI-Driven Attendance & Productivity Platform  +91 91360 96633  business@mano.co.in  B-11, 2nd Floor, ...
  3. Workforce Platform Smart Attendance & Workforce Management for Modern Teams  MANO-Attendance helps organizations track employee attendance, ...
